In [11]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_DIR = Path("..").resolve()

RAW_DIR  = PROJECT_DIR / "Data" / "Raw"
PROC_DIR = PROJECT_DIR / "Data" / "Processed"

assert RAW_DIR.exists(), f"RAW_DIR not found: {RAW_DIR}"
PROC_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("RAW_DIR    :", RAW_DIR)
print("PROC_DIR   :", PROC_DIR)

PROJECT_DIR: /Users/linda/RiceDatathon_2026_Finance
RAW_DIR    : /Users/linda/RiceDatathon_2026_Finance/Data/Raw
PROC_DIR   : /Users/linda/RiceDatathon_2026_Finance/Data/Processed


In [12]:
def load_prefer_clean(name_clean: str, name_raw: str) -> pd.DataFrame:
    p_clean = PROC_DIR / name_clean
    p_raw   = RAW_DIR / name_raw
    if p_clean.exists():
        return pd.read_csv(p_clean)
    return pd.read_csv(p_raw)

drv10 = load_prefer_clean("drv10_clean.csv", "master_panel_drv10.csv")
drv15 = load_prefer_clean("drv15_clean.csv", "master_panel_drv15.csv")
drv30 = load_prefer_clean("drv30_clean.csv", "master_panel_drv30.csv")

scoring = load_prefer_clean("scoring_clean.csv", "scoring.csv")

print("Loaded shapes:")
print("drv10  :", drv10.shape)
print("drv15  :", drv15.shape)
print("drv30  :", drv30.shape)
print("scoring:", scoring.shape)

Loaded shapes:
drv10  : (12889, 115)
drv15  : (13027, 115)
drv30  : (13025, 115)
scoring: (8997, 115)


In [13]:
def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = (
        out.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", "_", regex=True)
    )
    return out

drv10 = standardize_columns(drv10)
drv15 = standardize_columns(drv15)
drv30 = standardize_columns(drv30)
scoring = standardize_columns(scoring)

In [14]:
drv10 = drv10.copy(); drv10["drivetime_min"] = 10
drv15 = drv15.copy(); drv15["drivetime_min"] = 15
drv30 = drv30.copy(); drv30["drivetime_min"] = 30

panel = pd.concat([drv10, drv15, drv30], ignore_index=True)
print("panel:", panel.shape)
print("drivetime_min counts:\n", panel["drivetime_min"].value_counts(dropna=False))

panel: (38941, 116)
drivetime_min counts:
 drivetime_min
15    13027
30    13025
10    12889
Name: count, dtype: int64


In [15]:
DAILY_COLS = ["num_grocery_ta", "num_food_ta", "num_childcare_ta", "num_pharmacy_ta"]

LIFESTYLE_COLS = [
    "food_count_cafe", "food_count_bar_pub", "food_count_dessert",
    "food_count_fast_casual", "food_count_sit_down", "num_social_ta"
]

FAMILY_COLS = ["num_childcare_ta", "num_senior_ta", "food_count_sit_down"]

def ensure_numeric_fill0(df: pd.DataFrame, cols: list[str]) -> list[str]:
    present = [c for c in cols if c in df.columns]
    for c in present:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)
    return present

daily_present = ensure_numeric_fill0(panel, DAILY_COLS)
life_present  = ensure_numeric_fill0(panel, LIFESTYLE_COLS)
fam_present   = ensure_numeric_fill0(panel, FAMILY_COLS)

print("Daily inputs used:", daily_present)
print("Lifestyle inputs used:", life_present)
print("Family inputs used:", fam_present)

Daily inputs used: ['num_grocery_ta', 'num_food_ta', 'num_childcare_ta']
Lifestyle inputs used: ['food_count_cafe', 'food_count_bar_pub', 'food_count_dessert', 'food_count_fast_casual', 'food_count_sit_down', 'num_social_ta']
Family inputs used: ['num_childcare_ta', 'num_senior_ta', 'food_count_sit_down']


In [16]:
def z_within_group(df: pd.DataFrame, group_col: str, x_col: str) -> pd.Series:
    g = df.groupby(group_col)[x_col]
    mu = g.transform("mean")
    sd = g.transform(lambda s: s.std(ddof=0)).replace(0, np.nan)
    return ((df[x_col] - mu) / sd).fillna(0.0)

# Z for daily inputs only (primary)
for c in daily_present:
    panel[f"z_{c}"] = z_within_group(panel, "drivetime_min", c)

In [17]:
# Equal-weight daily score
z_daily_cols = [f"z_{c}" for c in daily_present]
panel["daily_needs_score_equal"] = panel[z_daily_cols].mean(axis=1) if z_daily_cols else 0.0

# Weighted daily score (uses only existing columns)
weights = {
    "num_grocery_ta": 0.30,
    "num_food_ta": 0.30,
    "num_childcare_ta": 0.20,
    "num_pharmacy_ta": 0.20,
}

panel["daily_needs_score"] = 0.0
for c, w in weights.items():
    zc = f"z_{c}"
    if zc in panel.columns:
        panel["daily_needs_score"] += w * panel[zc]

# Lifestyle / family as simple standardized sums (not z-scored unless you want)
panel["lifestyle_score"] = panel[life_present].sum(axis=1) if len(life_present) else 0.0
panel["family_support_score"] = panel[fam_present].sum(axis=1) if len(fam_present) else 0.0

panel[["daily_needs_score", "daily_needs_score_equal", "lifestyle_score", "family_support_score"]].describe()

,daily_needs_score,daily_needs_score_equal,lifestyle_score,family_support_score
count,3.894100e+04,3.894100e+04,38941.000000,38941.000000
mean,-1.113046e-17,-7.207426e-18,144.243394,91.235176
std,5.845592e-01,6.631490e-01,135.119669,81.215412
min,-1.212689e+00,-1.350945e+00,0.000000,0.000000
25%,-4.209778e-01,-4.697009e-01,54.000000,34.000000
50%,-1.055503e-01,-1.186954e-01,104.000000,67.000000
75%,3.627039e-01,4.023086e-01,185.000000,119.000000
max,5.377362e+00,9.490110e+00,915.000000,513.000000


In [18]:
sc = scoring.copy()

# drivetime
if "drivetime_min" not in sc.columns and "drivetime_minutes" in sc.columns:
    sc["drivetime_min"] = pd.to_numeric(sc["drivetime_minutes"], errors="coerce")

if "drivetime_min" not in sc.columns:
    raise RuntimeError("scoring is missing drivetime_min (or drivetime_minutes).")

# numeric inputs
_ = ensure_numeric_fill0(sc, DAILY_COLS)
_ = ensure_numeric_fill0(sc, LIFESTYLE_COLS)
_ = ensure_numeric_fill0(sc, FAMILY_COLS)

# z-score daily within ring
for c in daily_present:
    if c in sc.columns:
        sc[f"z_{c}"] = z_within_group(sc, "drivetime_min", c)
    else:
        sc[f"z_{c}"] = 0.0

z_daily_cols_sc = [f"z_{c}" for c in daily_present]
sc["daily_needs_score_equal"] = sc[z_daily_cols_sc].mean(axis=1) if z_daily_cols_sc else 0.0

sc["daily_needs_score"] = 0.0
for c, w in weights.items():
    zc = f"z_{c}"
    if zc in sc.columns:
        sc["daily_needs_score"] += w * sc[zc]

life_present_sc = [c for c in LIFESTYLE_COLS if c in sc.columns]
fam_present_sc  = [c for c in FAMILY_COLS if c in sc.columns]

sc["lifestyle_score"] = sc[life_present_sc].sum(axis=1) if len(life_present_sc) else 0.0
sc["family_support_score"] = sc[fam_present_sc].sum(axis=1) if len(fam_present_sc) else 0.0

scoring_scored = sc
print("scoring_scored:", scoring_scored.shape)

scoring_scored: (8997, 123)


In [19]:
panel_path  = PROC_DIR / "panel_with_amenity_scores.csv"
score_path  = PROC_DIR / "scoring_with_amenity_scores.csv"

panel.to_csv(panel_path, index=False)
scoring_scored.to_csv(score_path, index=False)

print("Saved:", panel_path)
print("Saved:", score_path)

Saved: /Users/linda/RiceDatathon_2026_Finance/Data/Processed/panel_with_amenity_scores.csv
Saved: /Users/linda/RiceDatathon_2026_Finance/Data/Processed/scoring_with_amenity_scores.csv


In [20]:
assert (PROC_DIR / "panel_with_amenity_scores.csv").exists()
assert (PROC_DIR / "scoring_with_amenity_scores.csv").exists()

print(panel[["drivetime_min","time_window_tag","daily_needs_score"]].head())

   drivetime_min time_window_tag  daily_needs_score
0             10            post          -0.118536
1             10            post           0.266642
2             10            post          -0.618611
3             10             pre          -0.562059
4             10            post          -0.570218
